In [3]:
import re
from pathlib import Path

import pandas as pd

In [4]:
paths = sorted(Path("output/decoders").rglob("eval_table.csv"))
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(attn|cross|crossreg)_reg([0-9]+)(_pep4)?", key)
    decoding, reg, pep = match.groups()
    table.insert(0, "pep", bool(pep))
    table.insert(0, "reg", int(reg))
    table.insert(0, "decoding", decoding)
    table.insert(0, "key", key)
    tables.append(table)
table = pd.concat(tables, ignore_index=True)
print(table.shape)
table.head()

136
(5860, 21)


,key,decoding,reg,pep,model,repr,clf,dataset,trial_id,C,...,acc,epoch,lr,wd,hparam_id,hparam,loss,acc_std,f1,f1_std
0,attn_reg1,attn,1,False,flat_mae,reg,logistic,aabc_age,0.0,0.359381,...,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,attn_reg1,attn,1,False,flat_mae,reg,logistic,aabc_age,1.0,0.046416,...,0.440476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,attn_reg1,attn,1,False,flat_mae,reg,logistic,aabc_age,2.0,0.005995,...,0.488095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,attn_reg1,attn,1,False,flat_mae,reg,logistic,aabc_age,3.0,0.046416,...,0.422619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,attn_reg1,attn,1,False,flat_mae,reg,logistic,aabc_age,4.0,0.046416,...,0.488095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
summary = table.loc[(table["split"] == "test") & (table["clf"] != "logistic")].pivot_table(
    values=["acc", "acc_std"],
    index=["decoding", "reg", "pep", "repr", "clf"],
    columns="dataset",
)
summary = summary.dropna(axis=1, how="all")
summary

acc                                    \
dataset                          aabc_sex hcpya_rest1lr_gender hcpya_task21   
decoding reg pep   repr  clf                                                  
attn     1   False patch attn    0.854545             0.836538     0.989683   
                   reg   linear  0.872727             0.807692     0.959127   
             True  patch attn    0.872727             0.798077     0.988492   
                   reg   linear  0.890909             0.846154     0.952579   
cross    1   False patch attn    0.909091             0.826923     0.987500   
                   reg   linear  0.890909             0.769231     0.914484   
             True  patch attn    0.872727             0.807692     0.989286   
                   reg   linear  0.909091             0.788462     0.893254   
crossreg 1   False patch attn    0.672727             0.673077     0.979167   
                   reg   linear  0.781818             0.605769     0.974206   
             True  patch attn    0.836364             0.817308     0.981746   
                   reg   linear  0.872727             0.711538     0.976190   
         4   False patch attn    0.890909             0.759615     0.984325   
                   reg   attn    0.800000             0.778846     0.978571   
                         linear  0.945455             0.769231     0.962500   
             True  patch attn    0.763636             0.817308     0.985119   
                   reg   linear  0.909091             0.817308     0.978373   
         16  False patch attn    0.854545             0.778846     0.982738   
                   reg   attn    0.909091             0.788462     0.976786   
                         linear  0.890909             0.759615     0.958135   

                                               acc_std                       \
dataset                         nsd_cococlip  aabc_sex hcpya_rest1lr_gender   
decoding reg pep   repr  clf                                                  
attn     1   False patch attn       0.305566  0.049770             0.034154   
                   reg   linear     0.117811  0.047458             0.035640   
             True  patch attn       0.310204  0.048812             0.039264   
                   reg   linear     0.106865  0.044098             0.034334   
cross    1   False patch attn       0.285529  0.041525             0.037665   
                   reg   linear     0.080334  0.043283             0.039071   
             True  patch attn       0.324119  0.048082             0.036059   
                   reg   linear     0.095362  0.041413             0.038961   
crossreg 1   False patch attn       0.294991  0.063833             0.043970   
                   reg   linear     0.244156  0.056361             0.048078   
             True  patch attn       0.307792  0.047066             0.038036   
                   reg   linear     0.245640  0.047083             0.043671   
         4   False patch attn       0.300000  0.043734             0.041365   
                   reg   attn       0.274768  0.053266             0.038708   
                         linear     0.132653  0.031540             0.038820   
             True  patch attn       0.313915  0.055481             0.036261   
                   reg   linear     0.205195  0.038779             0.035060   
         16  False patch attn       0.283117  0.050177             0.038360   
                   reg   attn       0.305380  0.038779             0.038265   
                         linear     0.105937  0.044098             0.038261   

                                                           
dataset                         hcpya_task21 nsd_cococlip  
decoding reg pep   repr  clf                               
attn     1   False patch attn       0.001413     0.005290  
                   reg   linear     0.002989     0.003619  
             True  patch attn       0.001447     0.005217  
                   reg   linear     0.0028

In [7]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


logistic_summary = table.loc[(table["split"] == "test") & (table["clf"] == "logistic")].pivot_table(
    values=["acc"],
    index=["decoding", "reg", "pep", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
logistic_summary = logistic_summary.loc[
    (slice(None), slice(None), slice(None), "reg", "logistic")
].dropna(axis=1, how="all")
logistic_summary

mean                                               \
                         acc                                                
dataset             aabc_age  aabc_sex  abide_dx adhd200_dx adni_ad_vs_cn   
decoding reg pep                                                            
attn     1   False  0.458750  0.872330  0.601935   0.612016      0.766260   
             True   0.443869  0.860568  0.588669   0.600388      0.760081   
cross    1   False  0.433631  0.847955  0.594073   0.623101      0.755935   
             True   0.432679  0.840511  0.556895   0.599225      0.758130   
crossreg 1   False  0.421369  0.743636  0.597823   0.570620      0.762520   
             True   0.418214  0.842216  0.608065   0.595814      0.770650   
         4   False  0.452560  0.852614  0.586210   0.609070      0.773740   
             True   0.444940  0.854091  0.599637   0.614806      0.774309   

                                                        sem            \
                                                        acc             
dataset            hcpya_rest1lr_gender   ppmi_dx  aabc_age  aabc_sex   
decoding reg pep                                                        
attn     1   False             0.879167  0.637688  0.003305  0.002285   
             True              0.855778  0.640955  0.003233  0.002511   
cross    1   False             0.845278  0.643317  0.003336  0.002341   
             True              0.831278  0.650754  0.003412  0.002519   
crossreg 1   False             0.705000  0.619749  0.003536  0.003014   
             True              0.817778  0.644372  0.003297  0.002076   
         4   False             0.846556  0.652462  0.003238  0.002318   
             True              0.827500  0.645678  0.003080  0.002319   

                                                                            \
                                                                             
dataset             abide_dx adhd200_dx adni_ad_vs_cn hcpya_rest1lr_gender   
decoding reg pep                                                             
attn     1   False  0.002942   0.003787      0.002419             0.002304   
             True   0.002919   0.004138      0.001973             0.002644   
cross    1   False  0.002807   0.003664      0.002021             0.002738   
             True   0.002869   0.003334      0.001619             0.002567   
crossreg 1   False  0.002540   0.003606      0.002199             0.002880   
             True   0.002618   0.003449      0.002372             0.002897   
         4   False  0.002887   0.003511      0.002605             0.002430   
             True   0.002991   0.003461      0.002385             0.002829   

                              
                              
dataset              ppmi_dx  
decoding reg pep              
attn     1   False  0.002469  
             True   0.002461  
cross    1   False  0.002447  
             True   0.002492  
crossreg 1   False  0.002113  
             True   0.002626  
         4   False  0.002516  
             True   0.002813